In [0]:
# läsa in bronze tabellen från bronze 
df = spark.table("marathos.bronze.raw_data")
display(df.limit(5))

In [0]:
df.select("Athlete_performance").show(20, truncate=False)

In [0]:
# Kolla schema på tabellen och antal rader
df.printSchema()
print(f"Antal rader: {df.count()}")

In [0]:
# Kolla unika enheter i Event_distance_length
df.select("Event_distance_length").distinct().display()

In [0]:
df.select("Athlete_performance").display()

In [0]:
display(df.select("Event_distance_length",
          "Athlete_performance").limit(50))

In [0]:
# importera funktioner
from pyspark.sql.functions import col

In [0]:
# Ta bort ogiltiga rader (behåll bara giltiga eventtyper km, mi, h)
df_clean = df.filter(
    col("Event_distance_length").endswith("km") |
    col("Event_distance_length").endswith("mi") |
    col("Event_distance_length").endswith("h") 
)

display(df_clean)


In [0]:
print(f"Antal rader efter första filtreringen: {df_clean.count()}")

In [0]:
# validera att event och performance matchar

df_valid = df_clean.filter(
    (col("Event_distance_length").like("%km%") | 
     col("Event_distance_length").like("%mi%")) & 
    col("Athlete_performance").like("%h%") |
    
    col("Event_distance_length").like("%h%") & 
    col("Athlete_performance").like("%km%")
)



In [0]:

print(f"Antal rader efter validering: {df_valid.count()}")

In [0]:
# Kontrollera resultatet (Kontrollera att event och performance matchar)
display(
df_valid.select("Athlete_performance", "Event_distance_length"))

In [0]:
# Skapa event_id för varje unikt event

from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

w = Window.orderBy("Event_name")

df_valid = df_valid.withColumn(
    "event_id", 
    dense_rank().over(w)
    )


In [0]:
df.columns

In [0]:
# Skapa cleaned tabell (One Big Table(OBT)) i silver layer 

df_obt = df_valid.select(
    "event_id",
    "Event_dates",
    "Event_name",
    "Event_distance_length",
    "Athlete_performance",
    "Athlete_country",
    "Athlete_gender",
    "Athlete_age_category",
    "Athlete_average_speed",
    "Athlete_ID"
)

In [0]:
# Spara tabellen i delta format till silver layer 

(df_obt.write
.format("delta")
.mode("overwrite").saveAsTable("marathos.silver.obt"))

In [0]:
display(df)